In [ ]:
# Cell 1 — install then force restart
!pip install -q --upgrade transformers peft trl bitsandbytes accelerate datasets huggingface_hub scikit-learn

import os
os.kill(os.getpid(), 9)  # force restart runtime

In [1]:
import transformers, peft, trl
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)

transformers: 5.8.0
peft: 0.19.1
trl: 1.4.0


In [1]:
import pandas as pd

# Most reliable current source
!pip install -q kaggle

# Alternative: use a hardcoded small financial sentiment dataset
# that we construct from known examples to guarantee it works
data = {
    "sentence": [
        "The company reported record profits this quarter.",
        "Sales declined significantly due to market conditions.",
        "The merger is expected to close in Q3.",
        "Operating costs increased by 15% year over year.",
        "The firm announced a special dividend for shareholders.",
        "Revenue fell short of analyst expectations.",
        "The acquisition will strengthen our market position.",
        "The company faces potential bankruptcy proceedings.",
        "Earnings per share rose by 20% compared to last year.",
        "The new product launch was met with disappointing sales.",
        "The board approved a share buyback programme.",
        "Net losses widened to £2.3 billion.",
        "The company secured a major contract worth £500 million.",
        "Credit ratings were downgraded by Moody's.",
        "Operating margin improved to 18% from 14%.",
        "The firm is under investigation for accounting irregularities.",
        "Customer growth exceeded targets for the third consecutive quarter.",
        "The company issued a profit warning for the full year.",
        "Return on equity reached its highest level in five years.",
        "The restructuring plan will result in 2,000 job losses.",
    ],
    "label_text": [
        "positive", "negative", "neutral", "negative", "positive",
        "negative", "positive", "negative", "positive", "negative",
        "positive", "negative", "positive", "negative", "positive",
        "negative", "positive", "negative", "positive", "negative",
    ]
}

# Load real dataset from HuggingFace spaces
try:
    import datasets
    ds = datasets.load_dataset("zeroshot/twitter-financial-news-sentiment")
    df = pd.DataFrame(ds["train"])
    label_map = {0: "bearish", 1: "bullish", 2: "neutral"}
    df["label_text"] = df["label"].map(label_map)
    # Remap to standard sentiment
    sentiment_map = {"bearish": "negative", "bullish": "positive", "neutral": "neutral"}
    df["label_text"] = df["label_text"].map(sentiment_map)
    df = df.rename(columns={"text": "sentence"})
    print(f"Loaded {len(df)} samples from Twitter Financial News")
    print(df["label_text"].value_counts())
    print(df.head())
except Exception as e:
    print(f"Fallback to manual dataset: {e}")
    df = pd.DataFrame(data)
    # Expand to 200 samples by repeating with slight variations
    df = pd.concat([df] * 10, ignore_index=True)
    print(f"Using manual dataset: {len(df)} samples")
    print(df["label_text"].value_counts())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded 9543 samples from Twitter Financial News
label_text
neutral     6178
positive    1923
negative    1442
Name: count, dtype: int64
                                            sentence  label label_text
0  $BYND - JPMorgan reels in expectations on Beyo...      0   negative
1  $CCL $RCL - Nomura points to bookings weakness...      0   negative
2  $CX - Cemex cut at Credit Suisse, J.P. Morgan ...      0   negative
3  $ESS: BTIG Research cuts to Neutral https://t....      0   negative
4  $FNKO - Funko slides after Piper Jaffray PT cu...      0   negative


In [3]:
from sklearn.model_selection import train_test_split

def format_prompt(row):
    return f"""### Instruction:
Analyse the sentiment of this financial text. Respond with exactly one word: positive, negative, or neutral.

### Input:
{row['sentence']}

### Response:
{row['label_text']}"""

df["text"] = df.apply(format_prompt, axis=1)

# Train/test split
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label_text"]
)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print("\nSample prompt:")
print(train_df.iloc[0]["text"])

Train: 7634 | Test: 1909

Sample prompt:
### Instruction:
Analyse the sentiment of this financial text. Respond with exactly one word: positive, negative, or neutral.

### Input:
Former Fed chief Bernanke sees bad year, no quick recovery #economy #MarketScreener https://t.co/YaN7qVv5aq https://t.co/zn3oEbjuL7

### Response:
negative


In [ ]:
import openai
from sklearn.metrics import classification_report

OPENAI_API_KEY = "sk-..."  # → OPENAI_API_KEY = "YOUR_OPENAI_KEY_HERE"
client = openai.OpenAI(api_key=OPENAI_API_KEY)

def gpt_predict(sentence):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": "You are a financial sentiment analyser. Respond with exactly one word: positive, negative, or neutral."
                },
                {"role": "user", "content": sentence}
            ],
            temperature=0,
            max_tokens=5,
        )
        pred = response.choices[0].message.content.strip().lower()
        pred = pred.replace(".", "").strip()
        return pred if pred in ["positive", "negative", "neutral"] else "neutral"
    except Exception:
        return "neutral"

# Run on 100 test samples — costs ~$0.02
test_sample = test_df.head(100).reset_index(drop=True)
print("Running GPT-4o-mini baseline on 100 samples...")

gpt_preds = []
for i, row in test_sample.iterrows():
    pred = gpt_predict(row["sentence"])
    gpt_preds.append(pred)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/100 done")

gpt_true = test_sample["label_text"].tolist()

print("\n--- GPT-4o-mini Baseline Results ---")
print(classification_report(gpt_true, gpt_preds))

# Save baseline scores for comparison later
import json
baseline_report = classification_report(gpt_true, gpt_preds, output_dict=True)
print(f"Baseline F1 (weighted): {baseline_report['weighted avg']['f1-score']:.3f}")

Running GPT-4o-mini baseline on 100 samples...
  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done

--- GPT-4o-mini Baseline Results ---
              precision    recall  f1-score   support

    negative       0.47      0.94      0.62        16
     neutral       0.97      0.51      0.67        68
    positive       0.47      0.94      0.62        16

    accuracy                           0.65       100
   macro avg       0.64      0.80      0.64       100
weighted avg       0.81      0.65      0.66       100

Baseline F1 (weighted): 0.658


In [3]:
# Quick data reload
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import pandas as pd

ds = load_dataset("zeroshot/twitter-financial-news-sentiment")
df = pd.DataFrame(ds["train"])
label_map = {0: "negative", 1: "bullish", 2: "neutral"}
df["label_text"] = df["label"].map(label_map)
df["label_text"] = df["label_text"].replace("bullish", "positive")
df = df.rename(columns={"text": "sentence"})

def format_prompt(row):
    return f"""### Instruction:
Analyse the sentiment of this financial text. Respond with exactly one word: positive, negative, or neutral.

### Input:
{row['sentence']}

### Response:
{row['label_text']}"""

df["text"] = df.apply(format_prompt, axis=1)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label_text"])
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

Train: 7634 | Test: 1909


In [20]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "microsoft/Phi-3.5-mini-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Phi-3.5 Mini with 4-bit quantisation...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
model = prepare_model_for_kbit_training(model)
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

print("All params cast to float16")
print("Verifying dtypes:")
dtypes = set(p.dtype for p in model.parameters())
print(dtypes)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading Phi-3.5 Mini with 4-bit quantisation...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

All params cast to float16
Verifying dtypes:
{torch.float32, torch.uint8}
trainable params: 3,145,728 || all params: 3,824,225,280 || trainable%: 0.0823


In [23]:
# Cast only LoRA weights to float16
for name, param in model.named_parameters():
    if 'lora' in name.lower() and param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

# Verify
bfloat_remaining = [(n, p.dtype) for n, p in model.named_parameters()
                     if p.dtype == torch.bfloat16]
print(f"BFloat16 layers remaining: {len(bfloat_remaining)}")

BFloat16 layers remaining: 0


In [26]:
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset as HFDataset
import torch
import numpy as np
from sklearn.metrics import classification_report

# Label encoding
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

tokenizer_bert = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer_bert(
        batch["sentence"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

# Prepare datasets
train_hf = HFDataset.from_pandas(
    train_df[["sentence", "label_text"]].reset_index(drop=True)
)
test_hf = HFDataset.from_pandas(
    test_df[["sentence", "label_text"]].reset_index(drop=True)
)

train_hf = train_hf.map(lambda x: {"label": label2id[x["label_text"]]})
test_hf = test_hf.map(lambda x: {"label": label2id[x["label_text"]]})

train_hf = train_hf.map(tokenize, batched=True)
test_hf = test_hf.map(tokenize, batched=True)

train_hf.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_hf.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load model
model_bert = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": (preds == labels).mean()}

training_args = TrainingArguments(
    output_dir="./fintone-distilbert",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    fp16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)

trainer_bert = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=test_hf,
    compute_metrics=compute_metrics,
)

print("Training FinTone-DistilBERT...")
print(f"Train: {len(train_hf)} | Test: {len(test_hf)}")
print("="*50)
trainer_bert.train()
print("Training complete!")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/7634 [00:00<?, ? examples/s]

Map:   0%|          | 0/1909 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


In [28]:
# Evaluate FinTone-DistilBERT vs GPT-4o-mini baseline
from sklearn.metrics import classification_report
import torch
import numpy as np

model_bert.eval()
test_sample = test_df.head(100).reset_index(drop=True)

fintone_preds = []
for i, row in test_sample.iterrows():
    inputs = tokenizer_bert(
        row["sentence"],
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    inputs = {k: v.to(model_bert.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model_bert(**inputs).logits
    pred_id = logits.argmax(-1).item()
    fintone_preds.append(id2label[pred_id])

fintone_true = test_sample["label_text"].tolist()

print("--- FinTone-DistilBERT Results ---")
print(classification_report(fintone_true, fintone_preds))

report = classification_report(fintone_true, fintone_preds, output_dict=True)
fintone_f1 = report["weighted avg"]["f1-score"]

print(f"\n{'='*40}")
print(f"GPT-4o-mini baseline F1:  0.658")
print(f"FinTone-DistilBERT F1:    {fintone_f1:.3f}")
print(f"Delta:                    {fintone_f1 - 0.658:+.3f}")
print(f"{'='*40}")

--- FinTone-DistilBERT Results ---
              precision    recall  f1-score   support

    negative       0.83      0.94      0.88        16
     neutral       0.97      0.90      0.93        68
    positive       0.74      0.88      0.80        16

    accuracy                           0.90       100
   macro avg       0.85      0.90      0.87       100
weighted avg       0.91      0.90      0.90       100


GPT-4o-mini baseline F1:  0.658
FinTone-DistilBERT F1:    0.902
Delta:                    +0.244


In [ ]:
from huggingface_hub import login

HF_TOKEN = "hf_..."        # → HF_TOKEN = "YOUR_HF_TOKEN_HERE"
login(token=HF_TOKEN)

model_bert.push_to_hub("Shun024/fintone-distilbert-financial-sentiment")
tokenizer_bert.push_to_hub("Shun024/fintone-distilbert-financial-sentiment")

print("Model pushed to HuggingFace Hub!")
print("View at: https://huggingface.co/Shun024/fintone-distilbert-financial-sentiment")

HfHubHTTPError: (Request ID: Root=1-6a020c3e-39b4b2cc0b7f99bd40d8d86f;fea3a114-6c66-4c93-8de9-6621a39bd34a)

403 Forbidden: You don't have the rights to create a model under the namespace "Shun024".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

In [ ]:
def fintone_predict(sentence, model, tokenizer):
    prompt = f"""### Instruction:
Analyse the sentiment of this financial text. Respond with exactly one word: positive, negative, or neutral.

### Input:
{sentence}

### Response:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    response = response.strip().lower().split()[0] if response.strip() else "neutral"
    if response not in ["positive", "negative", "neutral"]:
        response = "neutral"
    return response

print("Evaluating FinTone on test set...")
fintone_preds = []
for i, row in enumerate(test_sample.itertuples()):
    pred = fintone_predict(row.sentence, model, tokenizer)
    fintone_preds.append(pred)
    if (i+1) % 10 == 0:
        print(f"  {i+1}/100 done")

print("\nFinTone Results:")
print(classification_report(gpt_true, fintone_preds))

In [ ]:
from huggingface_hub import login

HF_TOKEN = "hf_..."        # → HF_TOKEN = "YOUR_HF_TOKEN_HERE"
login(token=HF_TOKEN)

model.push_to_hub("SLYM06/fintone-phi3-financial-sentiment")
tokenizer.push_to_hub("SLYM06/fintone-phi3-financial-sentiment")

print("Model pushed to HuggingFace Hub!")
print("View at: https://huggingface.co/SLYM06/fintone-phi3-financial-sentiment")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  20%|#9        | 1.25MB / 6.30MB            

README.md: 0.00B [00:00, ?B/s]

Model pushed to HuggingFace Hub!
View at: https://huggingface.co/SLYM06/fintone-phi3-financial-sentiment
